# Lab 3 — Build and serve the customized model with NVIDIA Riva

Lab 2 produced a complete NeMo `.nemo` checkpoint. Here, current Riva ServiceMaker performs the integrated NeMo-to-Riva conversion, builds a hardware-independent RMIR, and deploys an optimized model repository. Riva uses TensorRT and NVIDIA Triton internally, while applications call the supported Riva gRPC API.

Two deployment paths are provided:

1. **Amazon EKS (production path):** deploy the custom RMIR with the Riva Helm chart on a GPU-enabled EKS cluster.
2. **Local Docker on Brev (workshop path):** build, optimize, serve, and call the same Riva pipeline on this single GPU.


In [ ]:
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, time, wave

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'voice_asr_lab').exists():
    ROOT = ROOT.parent
if not (ROOT / 'src' / 'voice_asr_lab').exists():
    raise RuntimeError('Open this notebook from the workshop repository.')
sys.path.insert(0, str(ROOT / 'src'))

import riva.client
from jiwer import wer
from voice_asr_lab.audio import normalize_latin_text
from voice_asr_lab.nemo import read_nemo_manifest


## 1. Confirm the artifact and deployment controls

Riva 2.26.0 is pinned across the container, Helm chart, and Python client. The deploy phase must run on the target GPU because TensorRT engines are GPU-specific. The RMIR itself can be transferred to EKS.


In [ ]:
RIVA_VERSION = '2.26.0'
NEMO_MODEL = ROOT / 'artifacts' / 'parakeet-ctc-0.6b-nl.nemo'
RMIR_FILE = ROOT / 'artifacts' / 'riva' / 'own_your_voice_asr.rmir'
PIPELINE_NAME = 'own-your-voice-nl-asr-offline'
RIVA_URI = 'localhost:50051'
LANGUAGE_CODE = 'nl-NL'

if not NEMO_MODEL.is_file():
    raise RuntimeError(f'Missing {NEMO_MODEL}. Complete Lab 2 first.')
subprocess.run(['bash', str(ROOT / 'scripts' / 'stop_nim.sh')], check=True)
print({
    'nemo_model': str(NEMO_MODEL),
    'nemo_size_gb': round(NEMO_MODEL.stat().st_size / 1024**3, 2),
    'riva_version': RIVA_VERSION, 'pipeline': PIPELINE_NAME,
})


## 2. Build the Riva Model Intermediate Representation

The NVIDIA reference notebook shows standalone `nemo2riva`. Current Riva integrates that conversion into `riva-build`, so the recommended path passes the `.nemo` file directly and produces an RMIR. The key is hidden and supplied only to the subprocess environment.


In [ ]:
ngc_api_key = getpass('NGC API key (input is hidden): ').strip()
if not ngc_api_key:
    raise ValueError('An NGC API key is required for the Riva container.')
riva_env = {
    **os.environ, 'NGC_API_KEY': ngc_api_key, 'RIVA_VERSION': RIVA_VERSION,
    'NEMO_MODEL': str(NEMO_MODEL), 'RIVA_PIPELINE_NAME': PIPELINE_NAME,
}
subprocess.run(
    ['bash', str(ROOT / 'scripts' / 'build_riva_rmir.sh')],
    check=True, env=riva_env,
)
if not RMIR_FILE.is_file() or RMIR_FILE.stat().st_size == 0:
    raise RuntimeError('riva-build completed without a usable RMIR.')
print({'rmir': str(RMIR_FILE), 'size_gb': round(RMIR_FILE.stat().st_size / 1024**3, 2)})


## 3A. Production path — deploy the RMIR on Amazon EKS

This path assumes a pre-existing GPU-enabled EKS cluster; it is an instructor or platform-team exercise because creating AWS infrastructure is outside the attendee notebook. The Riva Helm chart stages the custom RMIR, runs GPU-specific optimization, creates the Triton repository, starts the Riva API, and exposes a Kubernetes Service.


In [ ]:
eks_guide = ROOT / 'deploy' / 'eks' / 'README.md'
values_override = ROOT / 'deploy' / 'eks' / 'values-custom-rmir.yaml'
print(eks_guide.read_text(encoding='utf-8'))
print('--- Helm model override ---')
print(values_override.read_text(encoding='utf-8'))


### EKS execution gate

The next cell performs read-only checks only when enabled. It never creates a cluster or installs Helm automatically. Use the guide after an AWS/EKS owner confirms the cluster, Region, GPU node capacity, storage class, NGC entitlement, and ingress policy.


In [ ]:
CHECK_EKS_PREREQUISITES = False
if CHECK_EKS_PREREQUISITES:
    for command in (
        ['aws', 'sts', 'get-caller-identity'],
        ['kubectl', 'cluster-info'],
        ['kubectl', 'get', 'nodes', '-o', 'wide'],
    ):
        subprocess.run(command, check=True)
else:
    print('EKS checks skipped. Use deploy/eks/README.md with the platform owner.')


## 3B. Workshop path — deploy Riva locally on the Brev GPU

This executes `riva-deploy` on the current GPU, starts the Riva ASR server, and mounts the generated model repository. First deployment can take many minutes. Port 50051 is the Riva gRPC boundary; the Triton ports remain internal.


In [ ]:
subprocess.run(
    ['bash', str(ROOT / 'scripts' / 'start_riva.sh')],
    check=True, env=riva_env,
)
del ngc_api_key
riva_env.pop('NGC_API_KEY', None)


## 4. Call the Riva gRPC API

We send raw 16-bit PCM from a held-out Dutch WAV and explicitly select the deployed pipeline. The same client works through `kubectl port-forward` for the EKS path.


In [ ]:
test_manifest = ROOT / 'artifacts' / 'nemo_manifests' / 'nl_test.jsonl'
test_rows = read_nemo_manifest(test_manifest)
sample = test_rows[0]
with wave.open(sample['audio_filepath'], 'rb') as wav_file:
    if wav_file.getsampwidth() != 2:
        raise RuntimeError('Riva client example expects 16-bit PCM WAV.')
    audio_bytes = wav_file.readframes(wav_file.getnframes())
    sample_rate = wav_file.getframerate()
    channels = wav_file.getnchannels()

auth = riva.client.Auth(uri=RIVA_URI)
asr_service = riva.client.ASRService(auth)
recognition_config = riva.client.RecognitionConfig(
    encoding=riva.client.AudioEncoding.LINEAR_PCM,
    sample_rate_hertz=sample_rate, audio_channel_count=channels,
    language_code=LANGUAGE_CODE, model=PIPELINE_NAME,
    max_alternatives=1, enable_automatic_punctuation=False,
)
response = asr_service.offline_recognize(audio_bytes, recognition_config)
if not response.results or not response.results[0].alternatives:
    raise RuntimeError('Riva returned no transcript alternatives.')
prediction = response.results[0].alternatives[0].transcript
reference = sample['text']
print({'reference': reference, 'prediction': prediction})


## 5. Measure service-boundary correctness and latency


In [ ]:
latencies = []
for _ in range(5):
    started = time.perf_counter()
    measured = asr_service.offline_recognize(audio_bytes, recognition_config)
    latencies.append(time.perf_counter() - started)
measured_prediction = measured.results[0].alternatives[0].transcript
audio_seconds = float(sample['duration'])
median_latency = sorted(latencies)[len(latencies) // 2]
service_report = {
    'sample_wer': wer(
        normalize_latin_text(reference), normalize_latin_text(measured_prediction)
    ),
    'audio_seconds': audio_seconds,
    'median_latency_seconds': median_latency,
    'real_time_factor': median_latency / audio_seconds,
    'throughput_x_realtime': audio_seconds / median_latency,
    'api': 'Riva gRPC', 'uri': RIVA_URI, 'model': PIPELINE_NAME,
}
report_path = ROOT / 'artifacts' / 'lab3_riva_report.json'
report_path.write_text(json.dumps(service_report, indent=2) + '\n', encoding='utf-8')
service_report


## Production handoff and cleanup

The local result proves the NeMo → RMIR → Riva API artifact chain on one GPU. It is not an EKS scale result. For production, optimize on the exact target GPU, keep nodes homogeneous when reusing model caches, use a shared PVC or the chart's S3 model cache, configure an HTTP/2/gRPC ingress with TLS, add health checks and telemetry, and load-test latency, throughput, concurrency, GPU memory, and held-out WER.

Stop the local service when finished: `bash scripts/stop_riva.sh`.
